### 1.1 Probing the article HTML scraping idea

In notebook 1.0 we found two limits in the raw corpus. Summaries are short, around 50 tokens, and the `source` label is misleading because Yahoo, which is 72% of the corpus, reposts other outlets' stories under its own name. We noted one idea that could fix both at once: follow each article's `url`, fetch the page HTML, and read the full body and the real publisher from it.

That was left as a direction worth testing rather than a settled plan. This notebook is that test. It is a probe, not a finished scraper. We take a small sample of articles from different sources, try to fetch each one, and record what comes back: whether the page loads, who really published it, and the time it states. We report what works, what does not, and why. Then we put the working part to use: we sort the sources into ones we can fully read and ones we cannot, count how much of the corpus that leaves us, recover the true first-publish time for the readable articles, and read a full body from each open source.

#### Setup

We load the raw corpus and bring in `requests` for the fetch and `BeautifulSoup` for reading the HTML. Both are already in the project, so no new dependency is added. We send a normal browser User-Agent header, because some sites answer differently, or not at all, to a bare script.

In [1]:
import time
import json
import datetime as dt

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

from stock_predictor.config import RAW_DATA_DIR

articles = pd.read_parquet(RAW_DATA_DIR / "raw_articles.parquet")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

print(f"Loaded {len(articles)} articles")

2026-08-10 16:04:34.645 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: C:\Users\adamw\programming-projects\Python\stock-predictor


Loaded 3143 articles


#### The stored url is a Finnhub redirect, not the article

The first thing to notice is that the `url` we saved is not a link to the outlet. Every row points at `finnhub.io/api/news?id=...`, a Finnhub redirect. That is useful rather than a problem: following the redirect lands on the real page, and the final host tells us where the story actually lives. This is our first clue that the true publisher is recoverable.

In [2]:
articles[["source", "url"]].head(5)

,source,url
0,Benzinga,https://finnhub.io/api/news?id=9cebcac6e8a3e8d...
1,Benzinga,https://finnhub.io/api/news?id=a80ce533d3eee6e...
2,SeekingAlpha,https://finnhub.io/api/news?id=7bb14df67bd5441...
3,Benzinga,https://finnhub.io/api/news?id=f798ed9f2cbdaff...
4,Yahoo,https://finnhub.io/api/news?id=6706e1dfdcf600a...


In [3]:
sample_url = articles.loc[articles["source"] == "Yahoo", "url"].iloc[0]
resp = requests.get(sample_url, headers=HEADERS, timeout=20)

print("stored url :", sample_url)
print("final url  :", resp.url)
print("final host :", resp.url.split("/")[2])

stored url : https://finnhub.io/api/news?id=6706e1dfdcf600a469bba901a5c73eab7669e16b3731b7d0802ce095bf24e133
final url  : https://finance.yahoo.com/news/analyst-tesla-tlsa-automotive-weakness-141330475.html
final host : finance.yahoo.com


#### A probe function

We write one function that fetches a single url and pulls back what the page can tell us: the HTTP status, the final host after redirects, the canonical link, the publisher named on the page, the published time, and how much body text we can read. Every failure is caught and returned as a value, so one dead link or block does not stop the run.

Two small helpers read the published time and the publisher. Most news sites embed a block of structured data marked `application/ld+json`, and a `datePublished` and `publisher` field usually sit inside it. Where that is missing we fall back to the common `article:published_time` and `og:site_name` meta tags.

In [4]:
def _published_time(soup):
    """Read the published time from JSON-LD, or fall back to a meta tag."""
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except json.JSONDecodeError:
            continue
        for obj in (data if isinstance(data, list) else [data]):
            if isinstance(obj, dict) and obj.get("datePublished"):
                return obj["datePublished"]
    meta = soup.find("meta", attrs={"property": "article:published_time"})
    return meta["content"] if meta and meta.get("content") else None


def _publisher(soup):
    """Read the real publisher named on the page."""
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except json.JSONDecodeError:
            continue
        for obj in (data if isinstance(data, list) else [data]):
            if isinstance(obj, dict):
                pub = obj.get("publisher")
                if isinstance(pub, dict) and pub.get("name"):
                    return pub["name"]
    meta = soup.find("meta", attrs={"property": "og:site_name"})
    return meta["content"] if meta and meta.get("content") else None


def _body_text(soup):
    """Join the page's paragraph text into one string."""
    return " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))


def probe(url):
    """Fetch one article url and report what the page reveals.

    Returns a dict with the status, final host, canonical link, publisher,
    published time, body length in characters, and any error caught.
    """
    out = {
        "status": None, "final_host": None, "canonical": None,
        "publisher": None, "published": None, "body_chars": 0, "error": None,
    }
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        out["status"] = r.status_code
        out["final_host"] = r.url.split("/")[2]
        if r.status_code != 200:
            return out
        soup = BeautifulSoup(r.text, "html.parser")
        link = soup.find("link", rel="canonical")
        if link and link.get("href"):
            out["canonical"] = link["href"]
        out["published"] = _published_time(soup)
        out["publisher"] = _publisher(soup)
        out["body_chars"] = len(_body_text(soup))
    except Exception as e:  # noqa: BLE001 - a probe should never crash on one url
        out["error"] = type(e).__name__
    return out

#### Running the probe across sources

We take a handful of articles from each of the eight sources and probe them. A short pause between requests keeps us polite and avoids tripping rate limits. The result is one row per article, so we can see at a glance which sources let us in, how much body we get, and who really published each piece.

In [5]:
sources = [
    "Yahoo", "Benzinga", "CNBC", "SeekingAlpha",
    "MarketWatch", "ChartMill", "DowJones", "Finnhub",
]

rows = []
for src in sources:
    subset = articles[articles["source"] == src]
    if subset.empty:
        continue
    for _, art in subset.sample(min(8, len(subset)), random_state=7).iterrows():
        res = probe(art["url"])
        rows.append({
            "source": src,
            "status": res["status"],
            "final_host": res["final_host"],
            "publisher": res["publisher"],
            "body_chars": res["body_chars"],
            "got_time": res["published"] is not None,
            "error": res["error"],
        })
        time.sleep(0.7)

probe_df = pd.DataFrame(rows)
probe_df

,source,status,final_host,publisher,body_chars,got_time,error
0,Yahoo,200,finance.yahoo.com,Yahoo! Finance,3738,True,None
1,Yahoo,200,www.fool.com,The Motley Fool,6551,True,None
2,Yahoo,200,finance.yahoo.com,Yahoo! Finance,1824,True,None
3,Yahoo,200,247wallst.com,24/7 Wall St.,4954,True,None
4,Yahoo,200,finance.yahoo.com,Yahoo! Finance,2804,True,None
...,...,...,...,...,...,...,...
59,Finnhub,200,finnhub.io,NaN,98,False,None
60,Finnhub,200,finnhub.io,NaN,2844,False,None
61,Finnhub,200,finnhub.io,NaN,98,False,None
62,Finnhub,200,finnhub.io,NaN,3601,False,None


#### How each source behaved

Grouping the probe by source gives a quick read on which outlets we can reach and how much text we get when we do. We look at the share of requests that returned a full 200 and the typical body length among those that did.

In [6]:
reached = probe_df["status"].eq(200)
ok_rows = probe_df[reached]

summary = pd.DataFrame({
    "tried": probe_df.groupby("source").size(),
    "reached_200": reached.groupby(probe_df["source"]).sum(),
    "median_body_chars": ok_rows.groupby("source")["body_chars"].median(),
}).fillna(0)
summary["reach_rate"] = summary["reached_200"] / summary["tried"]
summary.sort_values(["reach_rate", "median_body_chars"], ascending=False)

,tried,reached_200,median_body_chars,reach_rate
source,,,,
Benzinga,8,8,3493.5,1.000
DowJones,8,8,2420.5,1.000
CNBC,8,8,468.0,1.000
Finnhub,8,8,98.0,1.000
Yahoo,8,7,4954.0,0.875
SeekingAlpha,8,1,1512.0,0.125
ChartMill,8,0,0.0,0.000
MarketWatch,8,0,0.0,0.000


The sources fall into a clear picture. Three outlets come back with a full body of a few thousand characters, far more than the 50 token summary we had: Yahoo, Benzinga, and DowJones. Two answer with a 200 but little text. CNBC returns a page whose paragraph text is thin and inconsistent from one article to the next, because it loads much of its body with JavaScript that a plain fetch does not run, so a simple scrape often lands only a few hundred characters. Finnhub is worse still, returning near-empty stubs of under a hundred characters. The protected outlets do not let us in at all: SeekingAlpha and ChartMill return 403, and MarketWatch returns 401, sometimes after redirecting to Barron's, which lines up with the concern raised in 1.0 that consent walls and paywalls would be the main obstacle. SeekingAlpha's median body looks high only because a single page slipped through out of eight. The exact numbers shift a little between runs because the sample is random and the web is live, but the split itself is stable.

One caution on the body length. A high character count means we reached the page and read its paragraphs, not that the text is clean. Real extraction would still have to strip navigation, related-story links, and boilerplate, and it would break differently on each site's layout. So this is evidence the body is reachable on the open outlets, not that the cleaning problem is solved.

### The win: recovering the original source

The clearest success is the true publisher. In 1.0 the `source` label read Yahoo for 72% of the corpus, which hid who actually wrote each piece. Following the redirect and reading the page changes that. Below we take a handful of articles labelled Yahoo and print the Finnhub label next to the real host and the publisher named on the page.

In [7]:
yahoo = articles[articles["source"] == "Yahoo"].sample(6, random_state=3)

for _, art in yahoo.iterrows():
    res = probe(art["url"])
    print("headline             :", art["headline"][:70])
    print("finnhub source label :", art["source"])
    print("true host            :", res["final_host"])
    print("page publisher       :", res["publisher"])
    print()
    time.sleep(0.7)

headline             : DoorDash unveils Dot, its autonomous robot built to deliver your food
finnhub source label : Yahoo
true host            : finance.yahoo.com
page publisher       : Yahoo! Finance



headline             : Tesla Shares Spike on Reported SpaceX-xAI Merger Talks
finnhub source label : Yahoo
true host            : finance.yahoo.com
page publisher       : Yahoo! Finance



headline             : Opinion: 3 Main Drivers Will Define the Next Phase of the AI Race — an
finnhub source label : Yahoo
true host            : 247wallst.com
page publisher       : 24/7 Wall St.



headline             : Say Hello to the Tech Superstar That's Staring at a Multi-Trillion-Dol
finnhub source label : Yahoo
true host            : www.fool.com
page publisher       : The Motley Fool



headline             : How Wil Dow Jones Futures, Oil Prices React As U.S. Mulls Ground Troop
finnhub source label : Yahoo
true host            : finance.yahoo.com
page publisher       : Yahoo! Finance



headline             : Strategy's Bitcoin Accumulation Accelerates: More Upside Ahead?
finnhub source label : Yahoo
true host            : finance.yahoo.com
page publisher       : Yahoo! Finance



This works. Articles all labelled Yahoo turn out to come from a range of real hosts and publishers. Some stay on Yahoo Finance, but others resolve to outlets such as The Motley Fool or 24/7 Wall St., which the Yahoo label had flattened into one name. So the redirect plus the page gives us back the real source, which is exactly the first of the two limitations from 1.0. It is not perfect, since a few Yahoo links are dead and return a 404, but for the ones that load the true publisher comes through clearly.

### API release time against scraped release time

Since the fetch works on the open outlets, we can check one thing that matters a lot for the rest of the pipeline. Notebook 1.0 leaned on the article timestamp to line each story up with a market move, and warned that an error of a few hours would match an article to the wrong session and teach the model from wrong labels. Here we can test that timestamp against a second source, the time the page itself states.

For a sample of Yahoo, Benzinga, and CNBC articles we read the published time from the page, convert both times to UTC, and measure the gap in hours.

In [8]:
def to_utc(value):
    """Parse a page timestamp and return it in UTC."""
    parsed = date_parser.parse(value)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=dt.timezone.utc)
    return parsed.astimezone(dt.timezone.utc)


compare_sources = ["Yahoo", "Benzinga", "CNBC"]
records = []
for src in compare_sources:
    subset = articles[articles["source"] == src].sample(6, random_state=11)
    for _, art in subset.iterrows():
        res = probe(art["url"])
        if res["published"] is None:
            continue
        scraped = to_utc(res["published"])
        api_time = art["timestamp_utc"].to_pydatetime()
        records.append({
            "source": src,
            "api_utc": api_time,
            "scraped_utc": scraped,
            "delta_hours": round((scraped - api_time).total_seconds() / 3600, 2),
        })
        time.sleep(0.7)

compare_df = pd.DataFrame(records)
compare_df

,source,api_utc,scraped_utc,delta_hours
0,Yahoo,2026-05-28 12:39:12+00:00,2026-05-28 12:39:12+00:00,0.00
1,Yahoo,2025-08-23 18:46:23+00:00,2025-08-23 18:46:23+00:00,0.00
2,Yahoo,2026-02-27 18:53:01+00:00,2026-02-27 18:53:01+00:00,0.00
3,Yahoo,2026-07-31 02:16:52+00:00,2026-07-31 12:09:30+00:00,9.88
4,Yahoo,2026-02-26 13:22:59+00:00,2026-02-26 13:22:59+00:00,0.00
5,Yahoo,2026-03-31 17:38:09+00:00,2026-03-31 17:38:09+00:00,0.00
6,Benzinga,2026-04-29 02:17:53+00:00,2026-04-29 06:17:53+00:00,4.00
7,Benzinga,2026-07-30 14:56:25+00:00,2026-07-30 18:56:25+00:00,4.00
8,Benzinga,2025-12-23 05:32:14+00:00,2025-12-23 09:32:14+00:00,4.00
9,Benzinga,2026-03-30 08:20:13+00:00,2026-03-30 12:20:13+00:00,4.00


In [9]:
compare_df.groupby("source")["delta_hours"].agg(["mean", "min", "max", "count"])

,mean,min,max,count
source,,,,
Benzinga,4.000000,4.0,4.00,6
CNBC,4.500000,4.0,5.00,6
Yahoo,1.646667,0.0,9.88,6


This turns up a real and consistent problem. Benzinga is off by a steady 4 hours, and CNBC by 4 or 5 hours depending on the date. Those numbers are not random noise: 4 and 5 hours are exactly the US Eastern offset from UTC, 4 in summer under daylight time and 5 in winter, and in every case the page states a later UTC time than the API. Yahoo is the opposite, matching the page almost exactly at a gap of zero.

The reading is that for Benzinga and CNBC Finnhub took the publisher's local Eastern wall-clock time and stored it as if it were already UTC, so the timestamp it handed us is 4 to 5 hours too early. Yahoo matches because its own feed reports in UTC to begin with, so nothing was mislabelled. This is not a fault in our conversion in 1.0, which correctly reads a unix timestamp as UTC; the epoch value Finnhub provided is itself shifted for these sources.

The Yahoo column can carry the odd large outlier, a gap of several hours in a single row. That is a different effect from the steady Eastern shift: it comes from a wrapper page whose stated time is a later update, which we untangle in the true-time section below. It does not change the clear per-source pattern for Benzinga and CNBC.

This matters directly for the labelling step. An article Finnhub times at 13:00 UTC may really have gone out at 17:00 UTC. Matched to the market by the API time, it would land in the wrong part of the trading day, and for a close call it could be pushed onto the wrong side of the open or close. So the article timestamp cannot be trusted uniformly across sources, and the scraped page time is a way to check it and, where a page loads, to correct it.

### Sorting the sources: open against blocked

The probe now lets us put a label on each source. We call a source **open** if most of its requests reach a 200 and the typical body is long enough to be a real article, and **blocked** otherwise, which covers both paywalls that refuse us and sources that answer but return only a stub. We use a simple rule: at least half of requests reaching 200, and a median body of at least 1,000 characters. A stricter pipeline could tune these numbers, but they are enough to separate the two groups we already saw by eye.

In [10]:
THRESHOLD_CHARS = 1000
MIN_REACH = 0.5

access = summary.copy()

def classify(row):
    if row["reach_rate"] < MIN_REACH:
        return "blocked"
    if row["median_body_chars"] < THRESHOLD_CHARS:
        return "blocked"
    return "open"

access["label"] = access.apply(classify, axis=1)
access = access.sort_values("label")
access[["tried", "reached_200", "reach_rate", "median_body_chars", "label"]]

,tried,reached_200,reach_rate,median_body_chars,label
source,,,,,
CNBC,8,8,1.000,468.0,blocked
ChartMill,8,0,0.000,0.0,blocked
Finnhub,8,8,1.000,98.0,blocked
MarketWatch,8,0,0.000,0.0,blocked
SeekingAlpha,8,1,0.125,1512.0,blocked
Benzinga,8,8,1.000,3493.5,open
DowJones,8,8,1.000,2420.5,open
Yahoo,8,7,0.875,4954.0,open


In [11]:
open_sources = access.index[access["label"] == "open"].tolist()
blocked_sources = access.index[access["label"] == "blocked"].tolist()

print("open    :", open_sources)
print("blocked :", blocked_sources)

open    : ['Benzinga', 'DowJones', 'Yahoo']
blocked : ['CNBC', 'ChartMill', 'Finnhub', 'MarketWatch', 'SeekingAlpha']


### Counting the losses

With the sources labelled, we can count how much of the corpus we can fully read. We map every article to the label of its source and add up the two groups. This is a source-level count, so it treats every Yahoo article as open even though a few Yahoo links are dead or point onward to a paywalled outlet. It is the optimistic bound: the share we could reach if every open-source link behaved like the ones we sampled.

In [12]:
article_label = articles["source"].map(
    lambda s: "open" if s in open_sources else "blocked"
)

n_total = len(articles)
n_open = (article_label == "open").sum()
n_blocked = n_total - n_open

print(f"Fully accessible (open sources): {n_open:>5} of {n_total} ({n_open / n_total:.1%})")
print(f"Not fully accessible           : {n_blocked:>5} of {n_total} ({n_blocked / n_total:.1%})")
print()

breakdown = pd.DataFrame({"articles": articles["source"].value_counts()})
breakdown["label"] = breakdown.index.map(
    lambda s: "open" if s in open_sources else "blocked"
)
breakdown

Fully accessible (open sources):  2742 of 3143 (87.2%)
Not fully accessible           :   401 of 3143 (12.8%)



,articles,label
source,,
Yahoo,2274,open
Benzinga,451,open
SeekingAlpha,195,blocked
CNBC,145,blocked
ChartMill,24,blocked
MarketWatch,22,blocked
DowJones,17,open
Finnhub,15,blocked


The headline is encouraging: the great majority of the corpus, around seven in eight articles, sits on open sources, driven by Yahoo being both the largest source and a readable one. The loss is concentrated in a handful of outlets. SeekingAlpha is the largest single loss, a paywall we cannot pass. CNBC is the next, not because it blocks us but because its JavaScript body defeats a plain fetch, so under our rule it does not count as fully readable. The rest is the small ChartMill, MarketWatch, and Finnhub groups. Two caveats keep this honest. This counts at the source label, so the dead Yahoo 404s and the Yahoo reposts that lead on to a paywall are still counted as open, which means the true readable share is a little below the number here. And readable still does not mean clean: it means we can fetch the body, not that it is ready to score.

### Recovering the true first-publish time

Earlier we saw one Yahoo article whose page time was hours off the API time. The cause was a wrapper page: Yahoo hosts a short `/m/` page that carries its own repost or update time, while the real article lives elsewhere and its `canonical` link points there. To get the genuine first-publish time we follow that canonical link to the original publisher and read its `datePublished`, which by convention is the original time and is separate from `dateModified`, the edit time.

The function below does exactly that. It reads the time on the page we land on, finds the canonical link, and if that link points to a different host it fetches the original and reads its published time too. The original publisher's time is our best answer; the wrapper time is the fallback when there is no deeper source.

In [13]:
def _read_times(soup):
    """Return (published, modified) from JSON-LD, falling back to meta tags."""
    published = modified = None
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except json.JSONDecodeError:
            continue
        for obj in (data if isinstance(data, list) else [data]):
            if isinstance(obj, dict):
                published = published or obj.get("datePublished")
                modified = modified or obj.get("dateModified")
    if published is None:
        m = soup.find("meta", attrs={"property": "article:published_time"})
        published = m["content"] if m and m.get("content") else None
    if modified is None:
        m = soup.find("meta", attrs={"property": "article:modified_time"})
        modified = m["content"] if m and m.get("content") else None
    return published, modified


def original_publish(url):
    """Follow the canonical link to the original publisher and read its time.

    Returns the wrapper host and its published time, the canonical host, and the
    original published time. Where no deeper source exists, the wrapper time is
    used as the original.
    """
    out = {
        "wrapper_host": None, "wrapper_published": None,
        "canonical_host": None, "original_published": None, "error": None,
    }
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            out["error"] = f"status {r.status_code}"
            return out
        out["wrapper_host"] = r.url.split("/")[2]
        soup = BeautifulSoup(r.text, "html.parser")
        out["wrapper_published"], _ = _read_times(soup)

        link = soup.find("link", rel="canonical")
        canonical = link["href"] if link and link.get("href") else None
        if canonical:
            out["canonical_host"] = canonical.split("/")[2]
            if out["canonical_host"] != out["wrapper_host"]:
                r2 = requests.get(canonical, headers=HEADERS, timeout=20)
                if r2.status_code == 200:
                    orig, _ = _read_times(BeautifulSoup(r2.text, "html.parser"))
                    out["original_published"] = orig

        if out["original_published"] is None:
            out["original_published"] = out["wrapper_published"]
    except Exception as e:  # noqa: BLE001
        out["error"] = type(e).__name__
    return out

First we run it on the single article from before, the one whose Yahoo page time was hours adrift, to confirm the method recovers its real first-publish time.

In [14]:
outlier = (
    articles[articles["source"] == "Yahoo"]
    .sample(6, random_state=11)
    .reset_index(drop=True)
    .iloc[3]
)
res = original_publish(outlier["url"])

print("headline           :", outlier["headline"][:70])
print("api time (UTC)     :", outlier["timestamp_utc"])
print("wrapper host       :", res["wrapper_host"])
print("wrapper published  :", res["wrapper_published"])
print("canonical host     :", res["canonical_host"])
print("original published :", res["original_published"])

headline           : Elon Musk Mulls Selling Tesla's China Unit To Smooth SpaceX Merger: WS
api time (UTC)     : 2026-07-31 02:16:52+00:00
wrapper host       : finance.yahoo.com
wrapper published  : 2026-07-31T12:09:30Z
canonical host     : www.investors.com
original published : 2026-07-31T02:01:04+00:00


The wrapper on Yahoo stated a mid-morning time, but following the canonical link to the original publisher gives an early-morning first-publish time that sits right next to the API time. The wrapper time was the later update leaking through; the canonical page holds the real one.

Now we run the same pipeline across a sample of open-source articles and line up the API time against the recovered original time.

In [15]:
sample_open = articles[articles["source"].isin(open_sources)].sample(10, random_state=21)

rows = []
for _, art in sample_open.iterrows():
    res = original_publish(art["url"])
    original = to_utc(res["original_published"]) if res["original_published"] else None
    api_time = art["timestamp_utc"].to_pydatetime()
    rows.append({
        "source": art["source"],
        "api_utc": api_time,
        "wrapper_host": res["wrapper_host"],
        "canonical_host": res["canonical_host"],
        "original_utc": original,
        "delta_hours": round((original - api_time).total_seconds() / 3600, 2) if original else None,
    })
    time.sleep(0.7)

true_time_df = pd.DataFrame(rows)
true_time_df

,source,api_utc,wrapper_host,canonical_host,original_utc,delta_hours
0,Yahoo,2026-05-29 19:22:59+00:00,finance.yahoo.com,finance.yahoo.com,2026-05-29 19:22:59+00:00,0.00
1,Yahoo,2026-06-26 08:06:49+00:00,finance.yahoo.com,finance.yahoo.com,2026-06-26 08:06:49+00:00,0.00
2,Benzinga,2026-08-05 13:35:15+00:00,www.benzinga.com,www.benzinga.com,2026-08-05 17:35:15+00:00,4.00
3,Yahoo,2025-10-27 19:56:34+00:00,finance.yahoo.com,finance.yahoo.com,2025-10-27 19:56:34+00:00,0.00
4,Yahoo,2025-09-28 12:00:37+00:00,finance.yahoo.com,finance.yahoo.com,2025-09-28 12:00:37+00:00,0.00
5,Yahoo,2026-02-23 22:43:10+00:00,www.fool.com,www.fool.com,2026-02-23 22:23:10+00:00,-0.33
6,Yahoo,2025-11-26 21:00:55+00:00,finance.yahoo.com,finance.yahoo.com,2025-11-26 21:00:57+00:00,0.00
7,Yahoo,2026-01-30 15:00:41+00:00,finance.yahoo.com,finance.yahoo.com,2026-01-30 15:00:41+00:00,0.00
8,Yahoo,2025-10-27 13:30:00+00:00,finance.yahoo.com,finance.yahoo.com,2025-10-27 13:30:00+00:00,0.00
9,Yahoo,2025-10-29 04:51:14+00:00,finance.yahoo.com,www.investors.com,2025-11-05 16:12:14+00:00,179.35


Reading the table, the canonical follow mostly does its job. Where the wrapper host and the canonical host differ, as with the Yahoo reposts that resolve to fool.com or investors.com, we pick up the original publisher's own time, and for most rows the recovered time sits right on the API time or a clean 4 hours off it for Benzinga, the same Eastern shift as before. That shift is a fault in the API epoch rather than something the canonical can fix.

The method is not foolproof, and the table shows its weak spot. One row comes back hundreds of hours adrift, because the canonical link led to an investors.com page whose own stated time is a different, later article rather than the one we started from. So the canonical time is a strong second signal, not a guaranteed match: a small gap of zero or a clean Eastern offset can be trusted, but a large or odd gap is a flag to check the link rather than a correction to apply blindly. Even so, we now have two independent time signals per reachable article, the API time and the page time, which is exactly what we need to catch and correct the Eastern-shift error from 1.0.

### Reading a full body from each open source

Finally we pull the whole article body from one random article on each open source, to see the raw text the scrape actually returns. We skip any that come back too short, so a dead link does not stand in for a source, and print a long excerpt of each. This is the raw paragraph text, boilerplate and all, not a cleaned body.

In [16]:
def full_body(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return ""
    return _body_text(BeautifulSoup(r.text, "html.parser"))


def first_readable(src, tries=6, seed=99):
    """Return (article, body) for the first sampled article with real text."""
    subset = articles[articles["source"] == src].sample(min(tries, len(articles[articles["source"] == src])), random_state=seed)
    art = body = None
    for _, art in subset.iterrows():
        body = full_body(art["url"])
        if len(body) > 800:
            return art, body
        time.sleep(0.7)
    return art, body


for src in open_sources:
    art, body = first_readable(src)
    print("=" * 90)
    print(f"SOURCE: {src}")
    print("HEADLINE:", art["headline"])
    print(f"BODY LENGTH: {len(body)} chars (showing first 1500)")
    print("-" * 90)
    print(body[:1500])
    print()
    time.sleep(0.7)

SOURCE: Benzinga
HEADLINE: Trump Accounts App Hits Apple And Google Stores Thursday, Says Scott Bessent: 'Most Important Benefit Since...'
BODY LENGTH: 2893 chars (showing first 1500)
------------------------------------------------------------------------------------------
Treasury Secretary Scott Bessent said during Wednesday's Cabinet meeting that the Trump Accounts app will be available "on all major platforms tomorrow morning." Bessent called Trump Accounts "the most important government benefit for young people since the GI Bill" and said nearly 6 million children have already signed up for the program. Bessent compared the program to the GI Bill, the long-running U.S. benefits program that helps veterans pay for college, housing and job training. Trump Accounts are tax-deferred investment accounts for children created under President Donald Trump's One Big Beautiful Bill Act. The program gives eligible children a $1,000 government-funded investment account that families, employe

SOURCE: DowJones
HEADLINE: Stock Market Today: Dow Positive For November, But Nvidia Slides; Delayed Inflation Data Looms (Live Coverage)
BODY LENGTH: 2413 chars (showing first 1500)
------------------------------------------------------------------------------------------
TRENDING: Bullish Signals Amid 'Dirty' Volume  The Dow Jones Industrial Average and other major indexes held early gains all the way to the closing bell during an abbreviated trading session on Black Friday. Meanwhile, Nvidia (NVDA) was among the worst performers among both the Dow-30 and the Nasdaq-100 on the stock market today. Indexes were poised for hefty weekly gains as traders wrapped up a volatile… Copyright ©2026 Investor's Business Daily, LLC. All rights reserved. 87990cbe856818d5eddac44c7b1cdeb8 3:00 AM ET Nvidia takes a breather after jumping 12% last week. The stock is in a cup base with a buy point... 3:00 AM ET Nvidia takes a breather after jumping 12% last week. The... Artificial intelligence can help 

SOURCE: Yahoo
HEADLINE: Elon Musk Spent Months Trying to Cut Government Spending. Now He Says The Treasury Should Just “Issue People Checks”
BODY LENGTH: 5522 chars (showing first 1500)
------------------------------------------------------------------------------------------
Investing Tesla (TSLA) CEO Elon Musk pivoted from DOGE cost-cutter to advocate for Treasury checks, after federal outlays rose 6% during his tenure. Musk calls the concept "universal high income" rather than universal basic income, predicting AI will make work optional and drive deflation, not inflation. Consumer sentiment sits at 44.8, below the 60 recessionary threshold, and M2 money supply at $23 trillion directly challenges Musk's deflation forecast. Act now: the analyst who called NVIDIA in 2010 just named his top 10 AI stocks — and Tesla didn't make the cut. Grab the names FREE today . Sending You to Google News in 3 © 24/7 Wall St. / Getty Images In a new interview with The Economist’s Zanny Minton Beddoes,

The bodies confirm the point. On the open sources we get the full article text, many times longer than the Finnhub summary, and the real reporting is there to read. The excerpts also show the other half of the job still waiting: menu links, author blurbs, and related-story lines sit mixed in with the article, so a real run would need per-site rules to cut the body down to clean text before scoring.

### What this probe settled

**What worked**

- The stored Finnhub url redirects to the real page, so the true host and publisher are recoverable. This fixes the hidden-source limitation from 1.0 for the outlets we can reach.
- On the open outlets, Yahoo, Benzinga, and DowJones, the full article body is reachable, far longer than the summary we had.
- Around seven in eight articles sit on those open sources, so the reachable share is high and the loss is concentrated in a few outlets.
- Following the canonical link to the original publisher recovers the true first-publish time in most cases, which separates a genuine first publish from a later update on an aggregator wrapper.

**What did not**

- Paywalled and protected outlets, SeekingAlpha, ChartMill, and MarketWatch, which redirects to Barron's, block the fetch with 401 or 403 and return no article. This was the main risk called out in 1.0 and it is real.
- CNBC lets us in with a 200 but loads its body with JavaScript, so a plain fetch reads only a thin, inconsistent slice of the text. Getting its full body would need a headless browser. Finnhub links return near-empty stubs.
- Some Yahoo links are dead and return a 404, and some Yahoo reposts lead on to a paywall, so the true readable share is a little below the source-level count.
- The canonical follow can occasionally grab the wrong time when the canonical link points to a different or updated article, so large time gaps need a check rather than a blind correction.
- Reaching the page is not the same as clean text. The full bodies still carry navigation and boilerplate, so a tidy body would need per-site handling.

**The most useful finding** is the timestamp work. The API time is off by the Eastern offset for Benzinga and CNBC and correct for Yahoo, and the page gives us a second, checkable time, with the canonical follow recovering the genuine first-publish time in most cases. That is worth carrying forward on its own, separate from any full scraping effort, because it affects how every article is matched to the market.

On balance the scraping idea is promising but not free. It recovers the true source, reaches most of the corpus, and gives a way to fix the timestamp, which are real gains. It does not give clean full text without more work, it will never cover the paywalled sources, and the JavaScript outlets like CNBC need heavier tooling. A sensible next step is a small, source-aware fetch aimed first at the open outlets and at correcting timestamps, rather than a blanket scrape of the whole corpus.